# Module 03, Unit 02 — Directional Derivatives & Steepest Ascent

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 03 | Unit 02

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridge** | Gradient descent in logistic regression `[GLM]` |
| **Prerequisite** | Module 03, Unit 01 — The Gradient Vector |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] Define the directional derivative $D_{\boldsymbol{u}}f(\boldsymbol{x})$ and compute it as $\nabla f(\boldsymbol{x}) \cdot \boldsymbol{u}$
- [ ] Prove that the gradient gives the direction of steepest ascent using the Cauchy-Schwarz inequality
- [ ] State the steepest descent direction and the gradient descent update rule
- [ ] Explain the role of the learning rate $\alpha$ and describe what happens when it is too large or too small
- [ ] Write the binary cross-entropy loss for logistic regression and derive its gradient $\nabla_{\boldsymbol{\theta}} J$
- [ ] Trace one full gradient descent step in logistic regression from loss to parameter update

---

## Why This Unit

Unit 01 built the gradient as a geometric object. This unit puts it to work.

The central question is: *given a function $f$ and a point $\boldsymbol{x}$, in which direction should you move to change $f$ as rapidly as possible?* The answer — move in the direction of $\nabla f$ to increase, $-\nabla f$ to decrease — is the mathematical foundation of gradient-based optimization.

Every neural network trained by backpropagation, every logistic regression fit by an optimizer, every fine-tuned language model is executing a version of the update rule you will derive in this unit:

$$
\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \alpha\,\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta})
$$

Understanding *why* this rule works — not just that it does — is what separates practitioners who debug optimizers from those who can only run them.

---

## 1. The Directional Derivative

A partial derivative measures how $f$ changes in a coordinate direction — along $x_1$, along $x_2$, and so on. A **directional derivative** generalises this to an arbitrary direction.

**Definition.** Let $f : \mathbb{R}^n \to \mathbb{R}$ be differentiable at $\boldsymbol{x}$, and let $\boldsymbol{u} \in \mathbb{R}^n$ be a unit vector ($\|\boldsymbol{u}\| = 1$). The **directional derivative** of $f$ at $\boldsymbol{x}$ in the direction $\boldsymbol{u}$ is:

$$
D_{\boldsymbol{u}}f(\boldsymbol{x}) \coloneqq \lim_{h \to 0} \frac{f(\boldsymbol{x} + h\boldsymbol{u}) - f(\boldsymbol{x})}{h}
$$

This is the instantaneous rate of change of $f$ as you leave $\boldsymbol{x}$ in the direction $\boldsymbol{u}$ at unit speed.

> **Why unit vectors?** If $\boldsymbol{u}$ were not normalized, the directional derivative would depend on $\|\boldsymbol{u}\|$ as well as its direction. Requiring $\|\boldsymbol{u}\| = 1$ isolates the directional information and makes comparisons across directions meaningful.

**Theorem (Gradient formula).** If $f$ is differentiable at $\boldsymbol{x}$, then for any unit vector $\boldsymbol{u}$:

$$
D_{\boldsymbol{u}}f(\boldsymbol{x}) = \nabla f(\boldsymbol{x}) \cdot \boldsymbol{u}
$$

**Proof.** Define the single-variable function $g(h) = f(\boldsymbol{x} + h\boldsymbol{u})$. By the multivariable chain rule:

$$
g'(h) = \nabla f(\boldsymbol{x} + h\boldsymbol{u}) \cdot \boldsymbol{u}
$$

Evaluating at $h = 0$:

$$
D_{\boldsymbol{u}}f(\boldsymbol{x}) = g'(0) = \nabla f(\boldsymbol{x}) \cdot \boldsymbol{u} \qquad \square
$$

**Consequence.** The directional derivative is a single dot product. Once you have $\nabla f(\boldsymbol{x})$, you can compute the rate of change of $f$ in *any* direction with no additional differentiation.

**Worked Example 1.1.** Let $f(x, y) = x^2 + 2y^2$ and $\boldsymbol{u} = \left(\frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}}\right)^{\top}$ (the $45°$ direction).

$$
\nabla f(x, y) = \begin{pmatrix} 2x \\ 4y \end{pmatrix}
$$

At the point $(1, 1)$:

$$
D_{\boldsymbol{u}}f(1, 1) = \begin{pmatrix} 2 \\ 4 \end{pmatrix} \cdot \begin{pmatrix} 1/\sqrt{2} \\ 1/\sqrt{2} \end{pmatrix} = \frac{2}{\sqrt{2}} + \frac{4}{\sqrt{2}} = \frac{6}{\sqrt{2}} = 3\sqrt{2} \approx 4.24
$$

**Worked Example 1.2.** Same function and point, but now $\boldsymbol{u} = \left(-\frac{1}{\sqrt{2}}, \frac{1}{\sqrt{2}}\right)^{\top}$ (the $135°$ direction).

$$
D_{\boldsymbol{u}}f(1, 1) = \begin{pmatrix} 2 \\ 4 \end{pmatrix} \cdot \begin{pmatrix} -1/\sqrt{2} \\ 1/\sqrt{2} \end{pmatrix} = \frac{-2}{\sqrt{2}} + \frac{4}{\sqrt{2}} = \sqrt{2} \approx 1.41
$$

Moving in the $135°$ direction still increases $f$, but more slowly than in the $45°$ direction. Somewhere between $90°$ and $180°$ from the gradient, $D_{\boldsymbol{u}}f$ crosses zero — that is the tangent direction of the level curve.

---

## 2. Steepest Ascent and Steepest Descent

The dot product formula $D_{\boldsymbol{u}}f = \nabla f \cdot \boldsymbol{u}$ immediately raises a question: over all unit directions $\boldsymbol{u}$, which one maximizes $D_{\boldsymbol{u}}f$? Which one minimizes it?

**Theorem (Steepest directions).** Let $\nabla f(\boldsymbol{x}) \neq \boldsymbol{0}$. Among all unit vectors $\boldsymbol{u}$:

1. $D_{\boldsymbol{u}}f(\boldsymbol{x})$ is **maximized** by $\boldsymbol{u}^{*} = \dfrac{\nabla f(\boldsymbol{x})}{\|\nabla f(\boldsymbol{x})\|}$, with maximum value $\|\nabla f(\boldsymbol{x})\|$.
2. $D_{\boldsymbol{u}}f(\boldsymbol{x})$ is **minimized** by $\boldsymbol{u}^{*} = -\dfrac{\nabla f(\boldsymbol{x})}{\|\nabla f(\boldsymbol{x})\|}$, with minimum value $-\|\nabla f(\boldsymbol{x})\|$.
3. $D_{\boldsymbol{u}}f(\boldsymbol{x}) = 0$ for any $\boldsymbol{u}$ perpendicular to $\nabla f(\boldsymbol{x})$ — these are the tangent directions of the level set.

**Proof.** Write the dot product using the angle $\theta$ between $\nabla f$ and $\boldsymbol{u}$:

$$
D_{\boldsymbol{u}}f(\boldsymbol{x}) = \nabla f(\boldsymbol{x}) \cdot \boldsymbol{u} = \|\nabla f(\boldsymbol{x})\|\,\|\boldsymbol{u}\|\cos\theta = \|\nabla f(\boldsymbol{x})\|\cos\theta
$$

since $\|\boldsymbol{u}\| = 1$. Because $-1 \leq \cos\theta \leq 1$:

$$
-\|\nabla f(\boldsymbol{x})\| \leq D_{\boldsymbol{u}}f(\boldsymbol{x}) \leq \|\nabla f(\boldsymbol{x})\|
$$

The upper bound $\|\nabla f(\boldsymbol{x})\|$ is achieved when $\cos\theta = 1$, i.e., $\theta = 0$ — $\boldsymbol{u}$ points in the same direction as $\nabla f$.
The lower bound $-\|\nabla f(\boldsymbol{x})\|$ is achieved when $\cos\theta = -1$, i.e., $\theta = \pi$ — $\boldsymbol{u}$ points opposite to $\nabla f$. $\square$

This proof uses nothing beyond the definition of the dot product from Unit 02 of Module 00. The result is powerful: the gradient simultaneously encodes *which direction changes $f$ the most* and *how fast that maximum change is*.

**Summary of directional behavior:**

| Direction $\boldsymbol{u}$ relative to $\nabla f$ | $\cos\theta$ | $D_{\boldsymbol{u}}f$ | Interpretation |
|---|---|---|---|
| Same as $\nabla f$ ($\theta = 0°$) | $1$ | $\|\nabla f\|$ | Steepest ascent |
| Angle $\theta < 90°$ | $(0,1)$ | positive | $f$ increases |
| Perpendicular ($\theta = 90°$) | $0$ | $0$ | Along level set |
| Angle $\theta > 90°$ | $(-1,0)$ | negative | $f$ decreases |
| Opposite to $\nabla f$ ($\theta = 180°$) | $-1$ | $-\|\nabla f\|$ | Steepest descent |

---

## 3. The Gradient Descent Update Rule

In optimization we usually want to *minimize* a function — a loss, a cost, a negative log-likelihood. Steepest descent gives us a principled way to do this iteratively.

### 3.1 The Update Rule

Starting from a current point $\boldsymbol{\theta}^{(t)}$, take a small step in the direction of steepest descent:

$$
\boxed{\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \alpha\,\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}^{(t)})}
$$

where:
- $J(\boldsymbol{\theta})$ is the **loss function** (what we want to minimize)
- $\nabla_{\boldsymbol{\theta}}\,J$ is the **gradient** of the loss with respect to the parameters
- $\alpha > 0$ is the **learning rate** — a scalar step size

The update subtracts the gradient because we want to descend (minimize), not ascend.

### 3.2 Why This Decreases $J$ (Locally)

By the first-order Taylor approximation of $J$ at $\boldsymbol{\theta}^{(t)}$:

$$
J(\boldsymbol{\theta}^{(t+1)}) \approx J(\boldsymbol{\theta}^{(t)}) + \nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}^{(t)}) \cdot \left(\boldsymbol{\theta}^{(t+1)} - \boldsymbol{\theta}^{(t)}\right)
$$

Substituting the update $\boldsymbol{\theta}^{(t+1)} - \boldsymbol{\theta}^{(t)} = -\alpha\,\nabla_{\boldsymbol{\theta}}\,J$:

$$
J(\boldsymbol{\theta}^{(t+1)}) \approx J(\boldsymbol{\theta}^{(t)}) - \alpha\,\|\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}^{(t)})\|^2
$$

Since $\alpha > 0$ and $\|\nabla_{\boldsymbol{\theta}}\,J\|^2 \geq 0$, we have $J(\boldsymbol{\theta}^{(t+1)}) \leq J(\boldsymbol{\theta}^{(t)})$ — the loss decreases (or stays the same if the gradient is already zero). This holds for small enough $\alpha$; a step too large overshoots and the approximation breaks down.

### 3.3 The Role of the Learning Rate

The learning rate $\alpha$ is the most consequential hyperparameter in gradient-based optimization:

| Learning rate | Behavior |
|---|---|
| $\alpha$ too small | Convergence is correct but very slow; many iterations needed |
| $\alpha$ well-chosen | Smooth, efficient descent toward a minimum |
| $\alpha$ too large | Overshoots the minimum; loss may oscillate or diverge |

The second-order information (the Hessian, Module 04) can be used to choose $\alpha$ adaptively — this is the idea behind Newton's method and quasi-Newton optimizers like L-BFGS that are used to train statistical models.

> **Convergence.** Gradient descent converges to a local minimum when $\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}) = \boldsymbol{0}$. For **convex** loss functions — including the logistic regression loss below — every local minimum is the global minimum, so gradient descent finds the optimal parameters.

---

## 4. Statistical Bridge — Gradient Descent in Logistic Regression `[GLM]`

Logistic regression is the canonical example of a GLM that has no closed-form solution. Unlike ordinary least squares — where $\hat{\boldsymbol{\beta}} = (\boldsymbol{X}^{\top}\boldsymbol{X})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y}$ can be written explicitly — logistic regression parameters must be found by gradient-based optimization. Understanding this derivation concretely is the foundation for understanding how all classification models are trained.

### 4.1 Setup: Binary Classification

We have $n$ observations $(\boldsymbol{x}_i, y_i)$ where $\boldsymbol{x}_i \in \mathbb{R}^p$ is a feature vector and $y_i \in \{0, 1\}$ is a binary label. The logistic regression model is:

$$
p_i \coloneqq P(y_i = 1 \mid \boldsymbol{x}_i, \boldsymbol{\theta}) = \sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i) = \frac{1}{1 + e^{-\boldsymbol{\theta}^{\top}\boldsymbol{x}_i}}
$$

where $\sigma(z) = 1/(1 + e^{-z})$ is the **logistic sigmoid** function and $\boldsymbol{\theta} \in \mathbb{R}^p$ is the parameter vector to be learned.

### 4.2 The Loss Function

The **binary cross-entropy loss** (negative log-likelihood under the Bernoulli model) is:

$$
J(\boldsymbol{\theta}) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log p_i + (1 - y_i)\log(1 - p_i)\right]
$$

We want to **minimize** $J(\boldsymbol{\theta})$ over $\boldsymbol{\theta}$. This is equivalent to maximizing the log-likelihood — the $-1/n$ factor makes $J$ a loss (smaller is better) and normalizes by sample size.

### 4.3 Deriving the Gradient

This is the central calculation. We will derive $\nabla_{\boldsymbol{\theta}}\,J$ step by step.

**Step 1.** Write the loss contribution for a single observation $i$:

$$
J_i(\boldsymbol{\theta}) = -y_i \log \sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i) - (1 - y_i)\log(1 - \sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i))
$$

**Step 2.** Recall two facts from the single-variable review (Module 00, Unit 03):

$$
\sigma'(z) = \sigma(z)(1 - \sigma(z)) \qquad \frac{d}{dz}\log\sigma(z) = 1 - \sigma(z)
$$

**Step 3.** Apply the chain rule from Section 4.3 of Unit 01. For the composition $\log\sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i)$:

$$
\nabla_{\boldsymbol{\theta}}\,\log\sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i) = (1 - \sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i))\,\boldsymbol{x}_i = (1 - p_i)\,\boldsymbol{x}_i
$$

Similarly:

$$
\nabla_{\boldsymbol{\theta}}\,\log(1 - \sigma(\boldsymbol{\theta}^{\top}\boldsymbol{x}_i)) = -p_i\,\boldsymbol{x}_i
$$

**Step 4.** Combine:

$$
\nabla_{\boldsymbol{\theta}}\,J_i = -y_i(1-p_i)\boldsymbol{x}_i - (1-y_i)(-p_i)\boldsymbol{x}_i = \left[-y_i(1-p_i) + (1-y_i)p_i\right]\boldsymbol{x}_i
$$

Expanding the bracket:

$$
-y_i + y_i p_i + p_i - y_i p_i = p_i - y_i
$$

**Step 5.** Sum over all $n$ observations:

$$
\boxed{\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}) = \frac{1}{n}\sum_{i=1}^{n}(p_i - y_i)\,\boldsymbol{x}_i = \frac{1}{n}\boldsymbol{X}^{\top}(\hat{\boldsymbol{p}} - \boldsymbol{y})}
$$

where $\boldsymbol{X} \in \mathbb{R}^{n \times p}$ is the design matrix, $\hat{\boldsymbol{p}} = \sigma(\boldsymbol{X}\boldsymbol{\theta}) \in \mathbb{R}^n$ is the vector of predicted probabilities, and $\boldsymbol{y} \in \mathbb{R}^n$ is the label vector.

### 4.4 The Gradient Descent Step

Putting everything together, one gradient descent update for logistic regression is:

$$
\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \alpha \cdot \frac{1}{n}\boldsymbol{X}^{\top}\left(\hat{\boldsymbol{p}}^{(t)} - \boldsymbol{y}\right)
$$

**Interpretation.** The update is driven by the **residual vector** $\hat{\boldsymbol{p}}^{(t)} - \boldsymbol{y}$ — the difference between predicted probabilities and true labels. When the model is wrong ($\hat{p}_i$ far from $y_i$), the residuals are large, the gradient is large, and the update is large. As the model improves, residuals shrink, the gradient approaches zero, and updates become smaller — until convergence at $\nabla_{\boldsymbol{\theta}}\,J = \boldsymbol{0}$.

> **Connection to the score function.** Compare this to Unit 01: MLE sets $\nabla_{\boldsymbol{\theta}}\,\ell = \boldsymbol{0}$. Since $J = -\ell/n$, minimizing $J$ is equivalent to maximizing $\ell$, and gradient descent is iteratively solving the score equation $\nabla_{\boldsymbol{\theta}}\,\ell = \boldsymbol{0}$ by following the gradient. The two units are the same story told from opposite signs.

---

## Python: Visualization & Verification

The cells below demonstrate and visualize the concepts above. **Run them in order.** Modify the parameters and re-run — that is where the learning happens.

To run this notebook interactively, click the **rocket icon** at the top of the page and select **Open in Colab**.

In [ ]:
# ============================================================
# Install required libraries
# ============================================================
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy"]:
    install(pkg)

print("All packages installed.")

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import sympy as sp
from sympy import symbols, diff, cos, sin, sqrt, simplify, lambdify, pi

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'lines.linewidth': 2,
    'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')

### Section 1 — Directional Derivative as a Function of Angle

At a fixed point, how does $D_{\boldsymbol{u}}f$ vary as we rotate $\boldsymbol{u}$ through all directions? Since $D_{\boldsymbol{u}}f = \|\nabla f\|\cos\theta$, it is a cosine in the angle between $\boldsymbol{u}$ and $\nabla f$ — with a maximum in the gradient direction and a minimum in the opposite direction.

In [ ]:
# ============================================================
# Section 1 — Directional derivative vs angle (polar + Cartesian)
# ============================================================
# f(x,y) = x² + 2y²  at the point (1, 1)
# ∇f(1,1) = (2, 4)ᵀ,  ‖∇f‖ = √20

grad = np.array([2.0, 4.0])
grad_norm = np.linalg.norm(grad)
grad_angle = np.arctan2(grad[1], grad[0])   # angle of ∇f

phi = np.linspace(0, 2*np.pi, 500)           # angle of u
u   = np.column_stack([np.cos(phi), np.sin(phi)])   # unit vectors
Duf = u @ grad                               # D_u f = ∇f · u

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: polar plot of D_u f ---
ax = fig.add_subplot(121, projection='polar')
# Separate positive and negative parts for shading
pos_mask = Duf >= 0
neg_mask = Duf < 0

ax.plot(phi, np.abs(Duf), color=TS_BLUE, lw=2)
ax.fill_between(phi[pos_mask], 0, Duf[pos_mask],
                color=TS_BLUE, alpha=0.15, label='Ascending')
# For negative values, plot reflected
phi_neg = phi[neg_mask]
ax.fill_between(phi_neg, 0, np.abs(Duf[neg_mask]),
                color=TS_RED, alpha=0.15, label='Descending')

# Mark gradient direction
ax.annotate('', xy=(grad_angle, grad_norm), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=TS_AMBER, lw=2.5),
            xycoords='data', textcoords='data')
ax.text(grad_angle + 0.15, grad_norm + 0.3, r'$\nabla f$',
        color=TS_AMBER, fontsize=12, fontweight='bold')

ax.set_title(r'$D_{\mathbf{u}}f(1,1)$ vs direction of $\mathbf{u}$',
             pad=15, fontsize=11)

# --- Right: Cartesian — D_u f vs phi ---
ax2 = axes[1]
ax2.plot(np.degrees(phi), Duf, color=TS_BLUE, lw=2)
ax2.axhline(0, color=TS_GRAY, lw=1.0)
ax2.axhline(grad_norm, color=TS_AMBER, lw=1.5, ls='--',
            label=fr'Max = $\|\nabla f\|$ = {grad_norm:.2f}')
ax2.axhline(-grad_norm, color=TS_RED, lw=1.5, ls='--',
            label=fr'Min = $-\|\nabla f\|$ = {-grad_norm:.2f}')
ax2.fill_between(np.degrees(phi), Duf, 0, where=(Duf > 0),
                 alpha=0.12, color=TS_GREEN, label='f increasing')
ax2.fill_between(np.degrees(phi), Duf, 0, where=(Duf < 0),
                 alpha=0.12, color=TS_RED, label='f decreasing')

# Mark max and min
i_max = np.argmax(Duf)
i_min = np.argmin(Duf)
ax2.plot(np.degrees(phi[i_max]), Duf[i_max], 'o',
         color=TS_AMBER, markersize=9, zorder=5)
ax2.plot(np.degrees(phi[i_min]), Duf[i_min], 'o',
         color=TS_RED, markersize=9, zorder=5)

ax2.set_xlabel('Direction of $\\mathbf{u}$ (degrees)')
ax2.set_ylabel(r'$D_{\mathbf{u}}f(1,1)$')
ax2.set_title(r'Directional derivative: $\|\nabla f\|\cos\theta$')
ax2.legend(fontsize=9)
ax2.set_xticks(range(0, 361, 45))

plt.tight_layout()
plt.show()

print(f'Gradient direction: {np.degrees(grad_angle):.1f}°')
print(f'Max D_u f = ‖∇f‖ = {grad_norm:.4f}  (at {np.degrees(phi[i_max]):.1f}°)')
print(f'Min D_u f = −‖∇f‖ = {-grad_norm:.4f}  (at {np.degrees(phi[i_min]):.1f}°)')

**What do you see?**

- The directional derivative traces a perfect cosine as $\boldsymbol{u}$ rotates. Maximum at the gradient direction (~63.4°), minimum at the opposite direction (~243.4°), zero at the two perpendicular directions (±90° from the gradient).
- The polar plot shows the same information geometrically: the "lobe" in the gradient direction represents the ascending directions; the opposing region represents descent.

**Try modifying**: Change the point to $(2, 0)$. The gradient is now $(4, 0)^{\top}$ — purely horizontal. How does the plot change? What are the zero-crossing angles?

### Section 2 — Gradient Descent on a 2D Loss Surface

Visualize gradient descent as a path on a loss surface. We use a simple quadratic $J(\theta_1, \theta_2) = \theta_1^2 + 5\theta_2^2$ (an elongated bowl) to demonstrate how different learning rates affect convergence.

In [ ]:
# ============================================================
# Section 2 — Gradient descent paths at different learning rates
# ============================================================
def J(t1, t2):     return t1**2 + 5*t2**2
def dJ(t1, t2):    return np.array([2*t1, 10*t2])

def gradient_descent(t_init, lr, n_steps=60):
    path = [np.array(t_init, dtype=float)]
    t = np.array(t_init, dtype=float)
    for _ in range(n_steps):
        t = t - lr * dJ(t[0], t[1])
        path.append(t.copy())
        if np.linalg.norm(dJ(t[0], t[1])) < 1e-8:
            break
    return np.array(path)

t0 = [2.5, 1.8]
configs = [
    (0.02,  TS_GREEN, 'α = 0.02 (small — slow)'),
    (0.09,  TS_BLUE,  'α = 0.09 (well-chosen)'),
    (0.19,  TS_RED,   'α = 0.19 (too large — oscillates)'),
]

# Contour grid
t1g = np.linspace(-3, 3, 300)
t2g = np.linspace(-2.2, 2.2, 300)
T1, T2 = np.meshgrid(t1g, t2g)
Z = J(T1, T2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (lr, color, label) in zip(axes, configs):
    path = gradient_descent(t0, lr)
    loss_vals = J(path[:,0], path[:,1])

    cs = ax.contour(T1, T2, Z, levels=np.logspace(-2, 1.3, 18),
                    colors=[TS_GRAY], alpha=0.45, linewidths=0.8)
    ax.contourf(T1, T2, Z, levels=np.logspace(-2, 1.3, 18),
                cmap='Blues', alpha=0.18)

    ax.plot(path[:,0], path[:,1], 'o-', color=color,
            markersize=3, lw=1.5, alpha=0.85)
    ax.plot(*t0, 's', color=TS_GRAY, markersize=9, zorder=6,
            label='Start')
    ax.plot(0, 0, '*', color=TS_AMBER, markersize=14, zorder=6,
            label='Minimum')

    final_loss = loss_vals[-1]
    ax.set_title(f'{label}\n{len(path)-1} steps, J = {final_loss:.4f}',
                 fontsize=10)
    ax.set_xlabel(r'$\theta_1$'); ax.set_ylabel(r'$\theta_2$')
    ax.set_xlim(-3, 3); ax.set_ylim(-2.2, 2.2)
    ax.set_aspect('equal')
    ax.legend(fontsize=8, loc='upper right')

fig.suptitle(r'Gradient descent on $J(\theta_1, \theta_2) = \theta_1^2 + 5\theta_2^2$',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?**

- **Left (α = 0.02)**: The path descends smoothly but hugs the contours instead of cutting across them. Many small steps are needed.
- **Center (α = 0.09)**: The path converges efficiently, reaching the minimum in relatively few steps.
- **Right (α = 0.19)**: The path overshoots the minimum in the $\theta_2$ direction and oscillates. The elongated bowl (5× steeper in $\theta_2$) makes this asymmetry visible — the same learning rate that works well for $\theta_1$ is too large for $\theta_2$.

**This asymmetry is fundamental**: when loss surfaces have different curvatures in different directions (as all real ML loss functions do), a single fixed learning rate is always a compromise. This motivates adaptive learning rate methods (Adam, RMSProp) that maintain a separate effective $\alpha$ per parameter.

### Section 3 — Logistic Regression: Full Gradient Descent from Scratch

We implement the gradient derivation from Section 4 exactly as written and run it on synthetic 2D data. Watch the decision boundary move epoch by epoch as $\boldsymbol{\theta}$ converges.

In [ ]:
# ============================================================
# Section 3 — Logistic regression gradient descent from scratch
# ============================================================

# --- 1. Synthetic linearly separable data ---
rng = np.random.default_rng(7)
n_per_class = 60

X0 = rng.multivariate_normal([-1.5, -1.0], [[0.6, 0.1],[0.1, 0.6]], n_per_class)
X1 = rng.multivariate_normal([ 1.5,  1.0], [[0.6, 0.1],[0.1, 0.6]], n_per_class)
X_raw = np.vstack([X0, X1])  # (120, 2)
y   = np.hstack([np.zeros(n_per_class), np.ones(n_per_class)])

# Add bias column (intercept)
X = np.column_stack([np.ones(len(y)), X_raw])   # (120, 3)

# --- 2. Sigmoid and loss ---
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def cross_entropy(theta, X, y):
    p = sigmoid(X @ theta)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def gradient(theta, X, y):
    """∇J = (1/n) Xᵀ(p_hat − y)  — exactly as derived in Section 4.3"""
    p = sigmoid(X @ theta)
    return X.T @ (p - y) / len(y)

# --- 3. Gradient descent ---
theta = np.zeros(3)
lr    = 0.8
n_epochs = 120
loss_history = [cross_entropy(theta, X, y)]
theta_history = [theta.copy()]

for epoch in range(n_epochs):
    theta = theta - lr * gradient(theta, X, y)
    loss_history.append(cross_entropy(theta, X, y))
    theta_history.append(theta.copy())

theta_history = np.array(theta_history)

# --- 4. Decision boundary helper ---
def decision_boundary_x2(x1, theta):
    """Solve θ₀ + θ₁x₁ + θ₂x₂ = 0 for x₂"""
    return -(theta[0] + theta[1] * x1) / (theta[2] + 1e-12)

x1_line = np.linspace(-4, 4, 200)

# --- 5. Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: decision boundary evolution
ax = axes[0]
ax.scatter(X0[:,0], X0[:,1], color=TS_BLUE,  alpha=0.65, s=35,
           label='Class 0', zorder=3)
ax.scatter(X1[:,0], X1[:,1], color=TS_AMBER, alpha=0.65, s=35,
           label='Class 1', zorder=3)

# Draw boundaries at selected epochs
snapshot_epochs = [0, 5, 15, 40, n_epochs]
snapshot_colors = [TS_LIGHT, TS_GRAY, TS_GREEN, TS_RED, TS_BLUE]
for ep, col in zip(snapshot_epochs, snapshot_colors):
    th = theta_history[ep]
    x2_line = decision_boundary_x2(x1_line, th)
    mask = (x2_line > -4) & (x2_line < 4)
    lw   = 2.5 if ep == n_epochs else 1.2
    lab  = f'epoch {ep}' if ep > 0 else 'epoch 0 (random init)'
    ax.plot(x1_line[mask], x2_line[mask], color=col,
            lw=lw, ls='-', label=lab, alpha=0.9)

ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.set_title('Decision boundary convergence', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.set_aspect('equal')

# Right: loss curve
ax2 = axes[1]
ax2.plot(loss_history, color=TS_BLUE, lw=2)
for ep, col in zip(snapshot_epochs[1:], snapshot_colors[1:]):
    ax2.axvline(ep, color=col, lw=1.2, ls='--', alpha=0.8)
    ax2.text(ep + 1, loss_history[min(ep, len(loss_history)-1)] + 0.01,
             f'ep {ep}', color=col, fontsize=8)

ax2.set_xlabel('Epoch'); ax2.set_ylabel('Cross-entropy loss $J(\\theta)$')
ax2.set_title('Loss curve — gradient descent convergence', fontsize=12)

# Annotate gradient magnitude at start and end
g0  = np.linalg.norm(gradient(theta_history[0], X, y))
gfn = np.linalg.norm(gradient(theta_history[-1], X, y))
ax2.text(5, loss_history[5]*1.05,
         f'‖∇J‖ epoch 0: {g0:.3f}', fontsize=8, color=TS_GRAY)
ax2.text(n_epochs*0.6, loss_history[-1]*1.4,
         f'‖∇J‖ final: {gfn:.2e}', fontsize=8, color=TS_GRAY)

fig.suptitle(
    r'Logistic regression trained by gradient descent: $\nabla_{\theta}J = \frac{1}{n}\mathbf{X}^{\top}(\hat{\mathbf{p}} - \mathbf{y})$',
    fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print(f'Initial loss : {loss_history[0]:.4f}')
print(f'Final loss   : {loss_history[-1]:.4f}')
print(f'Final θ      : {theta}')
print(f'‖∇J‖ at convergence: {gfn:.2e}  (close to 0 ✓)')

**What do you see?**

- **Left**: The decision boundary rotates and translates over epochs. By epoch 40 it is essentially at its final position. The boundary is the set $\{\boldsymbol{x} : \boldsymbol{\theta}^{\top}\boldsymbol{x} = 0\}$ — a hyperplane in $\mathbb{R}^2$.
- **Right**: The loss drops steeply in the first few epochs then flattens as $\nabla_{\boldsymbol{\theta}} J \to \boldsymbol{0}$. The printed norm of the gradient at convergence confirms it is near zero — the update rule $\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \alpha \cdot \frac{1}{n}\boldsymbol{X}^{\top}(\hat{\boldsymbol{p}} - \boldsymbol{y})$ stopped producing meaningful changes.

**Key observation**: the gradient formula `X.T @ (p - y) / len(y)` in the code is a direct transcription of $\frac{1}{n}\boldsymbol{X}^{\top}(\hat{\boldsymbol{p}} - \boldsymbol{y})$ from Section 4.3. No library magic — just the derived formula executed as matrix operations.

**Try modifying**:
- Set `lr = 5.0`. Does the loss diverge or oscillate?
- Set `lr = 0.01`. How many epochs are needed to reach the same final loss?
- Change the data to be overlapping (reduce the class separation). What happens to the final boundary?

In [ ]:
# Extension workspace
# Suggestions:
# 1. Add L2 regularization to the loss: J_reg = J + (λ/2)‖θ‖²
#    Derive the modified gradient and implement it. How does the
#    decision boundary change for large λ?
#
# 2. Implement mini-batch gradient descent: instead of using all n
#    observations per update, sample a random batch of size 16.
#    Compare convergence to full-batch (this cell's implementation).
#
# 3. Compare convergence of gradient descent to Newton's method
#    (which uses the Hessian). We will cover the Hessian in Module 04,
#    but you can verify numerically that Newton converges in fewer steps.


---

## Summary

| Concept | Statement |
|---|---|
| Directional derivative | $D_{\boldsymbol{u}}f(\boldsymbol{x}) = \nabla f(\boldsymbol{x}) \cdot \boldsymbol{u}$, $\quad \|\boldsymbol{u}\| = 1$ |
| As a function of angle | $D_{\boldsymbol{u}}f = \|\nabla f\|\cos\theta$ where $\theta$ is angle between $\boldsymbol{u}$ and $\nabla f$ |
| Steepest ascent direction | $\boldsymbol{u}^* = \nabla f / \|\nabla f\|$, max rate $= \|\nabla f\|$ |
| Steepest descent direction | $\boldsymbol{u}^* = -\nabla f / \|\nabla f\|$, max decrease $= -\|\nabla f\|$ |
| Gradient descent update | $\boldsymbol{\theta}^{(t+1)} = \boldsymbol{\theta}^{(t)} - \alpha\,\nabla_{\boldsymbol{\theta}}\,J(\boldsymbol{\theta}^{(t)})$ |
| Logistic regression loss | $J(\boldsymbol{\theta}) = -\frac{1}{n}\sum_i\left[y_i \log p_i + (1-y_i)\log(1-p_i)\right]$ |
| Logistic regression gradient | $\nabla_{\boldsymbol{\theta}}\,J = \frac{1}{n}\boldsymbol{X}^{\top}(\hat{\boldsymbol{p}} - \boldsymbol{y})$ |
| Convergence condition | $\nabla_{\boldsymbol{\theta}}\,J(\hat{\boldsymbol{\theta}}) = \boldsymbol{0}$ |

**Up next:** Unit 03 — The Jacobian

We generalise the gradient to vector-valued functions $\boldsymbol{F} : \mathbb{R}^n \to \mathbb{R}^m$. The Jacobian matrix $\boldsymbol{J}_F$ collects all partial derivatives of all outputs, and the multivariable chain rule becomes matrix multiplication. The statistical bridge connects this directly to sensitivity analysis in multi-output neural networks.

---
*Module 03, Unit 02 — Threat Surfaces | fischer³ Education*